# 🏀 Miki Analítica — Tracker de Bàsquet

Detecta i segueix jugadores automàticament en vídeo de bàsquet.
Dos modes disponibles:

| Mode | Necessita | Resultat |
|------|-----------|----------|
| **basic** | Res (gratuït) | Tracking + estadístiques d'accions |
| **advanced** | Roboflow API key | + Metres recorreguts + Mapa de tir real + Mapa de calor |

| Escenari | Vídeo | API FCBQ | Resultat |
|----------|-------|----------|----------|
| **A** | ✅ | ✅ | Estadístiques completes |
| **B** | ✅ | ❌ | Tracking + mapes |
| **C** | ❌ | ✅ | Minuts reals jugats |

---
### ⚠️ Abans de començar
1. Activa la GPU: `Entorn d'execució → Canvia el tipus → T4 GPU`
2. Executa les cel·les **en ordre**
3. Espera ✅ a cada cel·la
4. El vídeo ha d'estar a **Google Drive**


## CEL·LA 1 — Instal·la les eines
⏱ ~3 minuts. Cal executar cada sessió nova.

In [ ]:
# Fix Pillow (cal fer-ho abans de tot)
!pip install -q 'Pillow==10.4.0'

# Instal·la la resta d'eines
!pip install -q ultralytics lapx opencv-python-headless yt-dlp

# Reinicia els mòduls de PIL per evitar errors
import importlib, PIL
importlib.reload(PIL)

import torch, cv2, numpy as np, pandas as pd
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO — activa la GPU!'
print(f'✅ Instal·lació completada')
print(f'   GPU: {gpu}')
print(f'   PyTorch: {torch.__version__} · OpenCV: {cv2.__version__}')


## CEL·LA 2 — Configuració
✏️ **Edita només aquesta cel·la.**

In [ ]:
# ══════════════════════════════════════════════════════════
# MODE: 'basic' = sense keypoints (gratuït, sempre funciona)
#       'advanced' = amb keypoints de pista (necessita API key)
MODE = 'basic'

# API key de Roboflow (només necessari si MODE = 'advanced')
# Obté-la a app.roboflow.com → Settings → API Keys
ROBOFLOW_API_KEY = 'POSA_AQUI_LA_API_KEY'

# ── Dades del partit ────────────────────────────────────────
ESCENARI       = 'A'   # 'A'=vídeo+API, 'B'=només vídeo, 'C'=només API
MATCH_ID       = '69ec95d4339c3d0001f523a1'
EQUIP_LOCAL    = 'Miki Lakers'
EQUIP_VISITANT = 'Mikinaikos'
NOM_SORTIDA    = 'dades_partit'

# Ruta del vídeo a Google Drive
VIDEO_PATH = '/content/drive/MyDrive/basquet/partit.mp4'

# Color de samarreta: 'clar' o 'fosc'
COLOR_LOCAL    = 'clar'
COLOR_VISITANT = 'fosc'

# Dimensions pista FIBA (metres)
PISTA_LLARG = 28.0
PISTA_AMPLE = 15.0

print(f'✅ Mode: {MODE.upper()}')
print(f'   Escenari {ESCENARI}: {EQUIP_LOCAL} vs {EQUIP_VISITANT}')
if MODE == 'advanced': print(f'   Roboflow API key: {ROBOFLOW_API_KEY[:8]}...')
if ESCENARI in ['A','C']: print(f'   Match ID: {MATCH_ID}')


## CEL·LA 3 — Connecta Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if ESCENARI in ['A','B']:
    if os.path.exists(VIDEO_PATH):
        cap=cv2.VideoCapture(VIDEO_PATH)
        FPS_VIDEO=cap.get(cv2.CAP_PROP_FPS)
        TOTAL_FRAMES=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        ret,FRAME_INICIAL=cap.read()
        FRAME_INICIAL_RGB=cv2.cvtColor(FRAME_INICIAL,cv2.COLOR_BGR2RGB)
        cap.release()
        DURACIO_SEG=TOTAL_FRAMES/FPS_VIDEO
        print(f'✅ Vídeo: {DURACIO_SEG/60:.1f} min · {FPS_VIDEO:.0f} fps · {TOTAL_FRAMES} frames')
        mida=os.path.getsize(VIDEO_PATH)/1024/1024/1024
        print(f'   Mida: {mida:.1f} GB')
    else:
        print(f'⚠️  Vídeo no trobat: {VIDEO_PATH}')
        print('   Comprova la ruta. Al panell esquerre: 📁 → drive → MyDrive → clic dret → Copia la ruta')
else:
    print('Escenari C: sense vídeo.')


## CEL·LA 4 — Descarrega dades de l'API FCBQ (escenaris A i C)

In [ ]:
import urllib.request, json

df_api = pd.DataFrame(); equip_noms = {}

if ESCENARI in ['A','C']:
    print('Descarregant API FCBQ...')
    url = f'https://msstats.optimalwayconsulting.com/v1/fcbq/getJsonWithMatchMoves/{MATCH_ID}?currentSeason=true'
    req = urllib.request.Request(url, headers={
        'User-Agent':'Mozilla/5.0','Accept':'application/json',
        'Referer':f'https://www.basquetcatala.cat/estadistiques/2025/{MATCH_ID}'})
    with urllib.request.urlopen(req, timeout=15) as r:
        data = json.loads(r.read())
    moves = data if isinstance(data, list) else data.get('moves',[])
    rows = []
    for m in moves:
        mn=m.get('min',0); sc=m.get('sec',0); q=m.get('period',1)
        rows.append({'temps_seg':round(((q-1)*10+mn+sc/60)*60,1),
            'quart':q,'min':mn,'sec':sc,
            'jugadora':m.get('actorName',''),
            'idEquip':str(m.get('idTeam','')),
            'accio':m.get('move',''),
            'idMove':m.get('idMove',0),
            'marcador':m.get('score','')})
    df_api = pd.DataFrame(rows)
    equips = [e for e in df_api['idEquip'].unique() if e and e!='0']
    if len(equips)>=2: equip_noms={equips[0]:EQUIP_LOCAL,equips[1]:EQUIP_VISITANT}
    df_api['equip_nom']=df_api['idEquip'].map(equip_noms).fillna('?')
    print(f'✅ {len(df_api)} jugades · {equip_noms}')
    subs=df_api[df_api['idMove'].isin([112,115])]
    print(f'   Substitucions: {len(subs)}')
else:
    print('Escenari B: sense API.')


## CEL·LA 5 — Minuts reals jugats (des de l'API)
**Escenari C**: aquí tens tot. Pots exportar i acabar.

In [ ]:
minuts_jugades = {}
if ESCENARI in ['A','C'] and not df_api.empty:
    DQ=600; intervals={}; en_pista={}
    for _,row in df_api.sort_values('temps_seg').iterrows():
        jug=row['jugadora']; t=row['temps_seg']; idm=row['idMove']; q=row['quart']
        if not jug: continue
        if idm==112: en_pista[jug]=t
        elif idm==115:
            ini=en_pista.pop(jug,(q-1)*DQ)
            intervals.setdefault(jug,[]).append((ini,t))
        elif idm==116:
            fi=q*DQ
            for j,ti in list(en_pista.items()): intervals.setdefault(j,[]).append((ti,fi))
            en_pista={}
    for jug,ivs in intervals.items(): minuts_jugades[jug]=round(sum(f-i for i,f in ivs)/60,1)
    df_min=pd.DataFrame([{'Jugadora':j,'Minuts':m,
        'Equip':df_api[df_api['jugadora']==j]['equip_nom'].iloc[0]
        if not df_api[df_api['jugadora']==j].empty else '?'}
        for j,m in sorted(minuts_jugades.items(),key=lambda x:-x[1])])
    print('✅ Minuts reals per jugadora:'); print(df_min.to_string(index=False))
    if ESCENARI=='C':
        from google.colab import files
        f=f'{NOM_SORTIDA}_minuts.csv'; df_min.to_csv(f,index=False)
        files.download(f); print(f'\n✅ Escenari C completat! CSV: {f}')
else:
    print('Sense API: minuts calculats des del vídeo.')


## CEL·LA 6 — Sincronització temps de joc ↔ vídeo
Busca al vídeo el moment on comença cada quart i apunta el segon.

In [ ]:
import numpy as np

# ── EDITA AQUÍ: segon del vídeo on comença cada quart ──────
ANCORATGES_VIDEO = {
    1: 45.0,    # Q1: l'àrbitre fa el salt
    2: 680.0,   # Q2
    3: 1290.0,  # Q3
    4: 1870.0,  # Q4
}
# ────────────────────────────────────────────────────────────

DQ_JOC=600; AJUSTOS_TL={}

def joc_a_video(t_joc, quart):
    if quart not in ANCORATGES_VIDEO: return t_joc
    prop=max(0,min(1,(t_joc-(quart-1)*DQ_JOC)/DQ_JOC))
    dur=(ANCORATGES_VIDEO[quart+1]-ANCORATGES_VIDEO[quart] if quart<4 else DQ_JOC*1.5)
    tv=ANCORATGES_VIDEO[quart]+prop*dur
    if AJUSTOS_TL:
        ts=sorted(AJUSTOS_TL)
        if tv<=ts[0]: tv+=AJUSTOS_TL[ts[0]]
        elif tv>=ts[-1]: tv+=AJUSTOS_TL[ts[-1]]
        else:
            for i in range(len(ts)-1):
                if ts[i]<=tv<=ts[i+1]:
                    p=(tv-ts[i])/(ts[i+1]-ts[i])
                    tv+=AJUSTOS_TL[ts[i]]+p*(AJUSTOS_TL[ts[i+1]]-AJUSTOS_TL[ts[i]]); break
    return round(tv,2)

print('✅ Sincronització configurada')
for q in sorted(ANCORATGES_VIDEO)[:-1]:
    at=ANCORATGES_VIDEO[q+1]-ANCORATGES_VIDEO[q]-DQ_JOC
    print(f'  Q{q}: ~{at:.0f}s d\'aturades ({at/60:.1f} min)')


## CEL·LA 7 — Calibratge fi amb tirs lliures (opcional)
Deixa `CALIBRATGE_TL = {}` buit per saltar.

In [ ]:
tirs_lliures_api=[]
if not df_api.empty:
    df_tl=df_api[df_api['accio'].str.contains('Cistella de 1|Intent fallat de 1',case=False,na=False)]
    for _,row in df_tl.iterrows():
        tirs_lliures_api.append({'jugadora':row['jugadora'],'quart':row['quart'],
            't_joc_fmt':f"Q{row['quart']} {int(row['min']):02d}:{int(row['sec']):02d}",
            't_video_est':round(joc_a_video(row['temps_seg'],row['quart']),1)})
print(f'Tirs lliures a l\'API: {len(tirs_lliures_api)}')
for i,tl in enumerate(tirs_lliures_api[:8]):
    print(f'  TL {i+1:2d}: {tl["t_joc_fmt"]:12s} {tl["jugadora"]:28s} ~ {tl["t_video_est"]:.0f}s')

# ── EDITA AQUÍ si vols calibrar: {numero_TL: segon_real_al_video} ──
CALIBRATGE_TL = {}
# ──────────────────────────────────────────────────────────────────

AJUSTOS_TL={}
if CALIBRATGE_TL:
    for idx1,t_real in CALIBRATGE_TL.items():
        idx=idx1-1
        if idx<len(tirs_lliures_api):
            tl=tirs_lliures_api[idx]; ajust=t_real-tl['t_video_est']
            AJUSTOS_TL[tl['t_video_est']]=ajust
            print(f'  TL {idx1}: ajust {ajust:+.1f}s')
    print(f'✅ Calibratge amb {len(AJUSTOS_TL)} punts')
else:
    print('ℹ️  Sense calibratge — interpolació lineal')


## CEL·LA 8 — Carrega els models
Mode basic: YOLOv8. Mode advanced: YOLOv8 + model de keypoints de pista.
⏱ ~2-3 minuts

In [ ]:
from ultralytics import YOLO
import torch, os

# Model de tracking (sempre)
model_tracking = YOLO('yolov8n.pt')
print(f'✅ Model tracking carregat (YOLOv8)')

# Model de keypoints (només mode advanced)
model_keypoints = None
USAR_KEYPOINTS = False

if MODE == 'advanced':
    print('Carregant model de keypoints de pista...')
    !pip install -q inference
    from inference import get_model
    os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
    try:
        model_keypoints = get_model(
            model_id='basketball-court-detection-2/13',
            api_key=ROBOFLOW_API_KEY
        )
        # Prova amb el primer frame
        result_test = model_keypoints.infer(FRAME_INICIAL_RGB)[0]
        kps_test = result_test.predictions if hasattr(result_test,'predictions') else []
        print(f'✅ Model keypoints carregat! Deteccions test: {len(kps_test)}')
        print('   S\'executa localment — 0 crèdits Roboflow gastats per frame.')
        USAR_KEYPOINTS = True
    except Exception as e:
        print(f'⚠️  Error keypoints: {e}')
        print('   Continuant en mode basic.')
        USAR_KEYPOINTS = False
else:
    print('ℹ️  Mode basic: sense keypoints de pista.')
    print('   Canvia MODE="advanced" a la cel·la 2 per activar-los.')


## CEL·LA 9 — Detecta les cantonades de la pista
Mode basic: detecció automàtica per línies. Mode advanced: keypoints del model.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches

CAMERA_FIXA = False
HOMOGRAFIA  = None

def pixels_a_metres(px, py, H=None):
    h = H if H is not None else HOMOGRAFIA
    if h is None: return None, None
    p=np.array([[[float(px),float(py)]]],dtype=np.float32)
    r=cv2.perspectiveTransform(p,h)
    return round(float(np.clip(r[0][0][0],0,PISTA_LLARG)),2),\
           round(float(np.clip(r[0][0][1],0,PISTA_AMPLE)),2)

if ESCENARI in ['A','B']:
    if MODE == 'advanced' and USAR_KEYPOINTS:
        # ── Mode advanced: keypoints del model ────────────────────────
        COURT_KEYPOINTS_REALS = {
            0:(0.0,0.0), 1:(28.0,0.0), 2:(28.0,15.0), 3:(0.0,15.0),
            4:(14.0,0.0), 5:(14.0,15.0), 6:(5.8,4.5), 7:(5.8,10.5),
            8:(22.2,4.5), 9:(22.2,10.5)
        }
        def detecta_kps(frame_rgb):
            if model_keypoints is None: return None
            try:
                result=model_keypoints.infer(frame_rgb)[0]
                if not hasattr(result,'predictions') or not result.predictions: return None
                pred=max(result.predictions,key=lambda x: x.confidence)
                return pred.keypoints if hasattr(pred,'keypoints') else None
            except: return None

        def calcula_H(kps):
            if not kps or len(kps)<4: return None
            pi,pr=[],[]
            for i,kp in enumerate(kps):
                c=kp.confidence if hasattr(kp,'confidence') else 0
                if i in COURT_KEYPOINTS_REALS and c>0.3:
                    pi.append([kp.x,kp.y]); pr.append(list(COURT_KEYPOINTS_REALS[i]))
            if len(pi)<4: return None
            H,_=cv2.findHomography(np.float32(pi),np.float32(pr),cv2.RANSAC,5.0)
            return H

        kps_ini = detecta_kps(FRAME_INICIAL_RGB)
        H_ini   = calcula_H(kps_ini) if kps_ini else None
        if H_ini is not None:
            CAMERA_FIXA=True; HOMOGRAFIA=H_ini
            n=len([k for k in kps_ini if (k.confidence if hasattr(k,'confidence') else 0)>0.3])
            print(f'✅ Mode advanced: {n} keypoints · Homografia calculada!')
            print('   Coordenades reals disponibles fins i tot amb càmera mòbil!')
        else:
            print('ℹ️  Model no detecta prou keypoints. Provant detecció automàtica...')

    if not CAMERA_FIXA:
        # ── Mode basic (o fallback): detecció automàtica per línies ───
        gray=cv2.cvtColor(FRAME_INICIAL,cv2.COLOR_BGR2GRAY)
        edges=cv2.Canny(cv2.GaussianBlur(gray,(5,5),0),50,150)
        lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=100,
                              minLineLength=FRAME_INICIAL.shape[1]//4,maxLineGap=20)
        cantonades=None
        if lines is not None and len(lines)>=4:
            horitz=[l[0] for l in lines if abs(l[0][1]-l[0][3])<20]
            vertic=[l[0] for l in lines if abs(l[0][0]-l[0][2])<20]
            if len(horitz)>=2 and len(vertic)>=2:
                hy=sorted([min(l[1],l[3]) for l in horitz])
                vx=sorted([min(l[0],l[2]) for l in vertic])
                y_t,y_b=hy[0],hy[-1]; x_l,x_r=vx[0],vx[-1]
                h,w=FRAME_INICIAL.shape[:2]
                if (x_r-x_l)>=w*0.5 and (y_b-y_t)>=h*0.4:
                    cantonades=np.float32([[x_l,y_t],[x_r,y_t],[x_r,y_b],[x_l,y_b]])
        if cantonades is not None:
            CAMERA_FIXA=True
            HOMOGRAFIA,_=cv2.findHomography(cantonades,
                np.float32([[0,PISTA_AMPLE],[PISTA_LLARG,PISTA_AMPLE],[PISTA_LLARG,0],[0,0]]))
            print('✅ CÀMERA FIXA detectada automàticament!')
        else:
            print('ℹ️  CÀMERA MÒBIL — sense coordenades reals.')
            print('   Per activar coordenades: canvia MODE="advanced" i posa la API key.')

    # Resum funcionalitats
    print()
    print('Funcionalitats disponibles:')
    print(f'  {"✅" if True else "❌"} Tracking de jugadores')
    print(f'  {"✅" if not df_api.empty else "❌"} Estadístiques d\'accions (API)')
    print(f'  {"✅" if CAMERA_FIXA else "❌"} Metres recorreguts')
    print(f'  {"✅" if CAMERA_FIXA else "❌"} Mapa de tir real')
    print(f'  {"✅" if CAMERA_FIXA else "❌"} Mapa de calor de posicions')

    # Visualitza el primer frame
    fig,ax=plt.subplots(figsize=(12,7))
    ax.imshow(FRAME_INICIAL_RGB); ax.axis('off')
    color_titol='#16a34a' if CAMERA_FIXA else '#d97706'
    ax.set_title(f'{"CÀMERA FIXA ✅" if CAMERA_FIXA else "CÀMERA MÒBIL ⚠️"} — Mode {MODE.upper()}',
                 fontsize=12, color=color_titol)
    plt.tight_layout(); plt.show()
else:
    print('Escenari C: sense vídeo.')
    CAMERA_FIXA=False
    def detecta_kps(x): return None
    def calcula_H(x): return None


## CEL·LA 9b — Cantonades manuals (opcional)
Executa NOMÉS si la cel·la 9 ha dit 'càmera mòbil' però el vídeo SÍ és de càmera fixa.

In [ ]:
# ── EDITA AQUÍ: coordenades en píxels dels 4 cantons ───────
# Mira el frame de dalt i apunta les cantonades de la pista
#
#  CANT_1 ─────── CANT_2
#    │    PISTA     │
#  CANT_4 ─────── CANT_3
#
CANTONADES_MANUAL = np.float32([
    [95,   60],   # Superior esquerra
    [1185, 60],   # Superior dreta
    [1185, 660],  # Inferior dreta
    [95,   660],  # Inferior esquerra
])
# ────────────────────────────────────────────────────────────

CAMERA_FIXA = True
HOMOGRAFIA, _ = cv2.findHomography(
    CANTONADES_MANUAL,
    np.float32([[0,PISTA_AMPLE],[PISTA_LLARG,PISTA_AMPLE],[PISTA_LLARG,0],[0,0]]))
print('✅ Cantonades manuals aplicades')
print(f'  Test: {pixels_a_metres(*CANTONADES_MANUAL[3])} (ha de ser 0,0)')
print(f'  Test: {pixels_a_metres(*CANTONADES_MANUAL[1])} (ha de ser {PISTA_LLARG},{PISTA_AMPLE})')


## CEL·LA 10 — Tracking amb YOLOv8 + ByteTrack
⏱ 15-30 minuts per a un partit de 90 min amb GPU T4.

In [ ]:
import time
from tqdm import tqdm

print(f'Mode: {MODE.upper()} · Càmera: {"FIXA" if CAMERA_FIXA else "MÒBIL"}')

# Si mode advanced, actualitza keypoints cada N frames
KEYPOINT_INTERVAL = 30  # frames de tracking
H_actual = HOMOGRAFIA

cap     = cv2.VideoCapture(VIDEO_PATH)
fps_vid = cap.get(cv2.CAP_PROP_FPS)
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
INTERVAL = max(1, int(fps_vid/5))  # 5 fps de tracking
print(f'Processant {total//INTERVAL} frames...')

resultats=[]; frame_idx=0; proc_idx=0; t0=time.time()

with tqdm(total=total//INTERVAL) as pbar:
    while cap.isOpened():
        ret,frame=cap.read()
        if not ret: break
        if frame_idx%INTERVAL==0:
            temps_seg=frame_idx/fps_vid

            # Actualitza keypoints (mode advanced)
            if MODE=='advanced' and USAR_KEYPOINTS and proc_idx%KEYPOINT_INTERVAL==0:
                frame_rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
                kps_f=detecta_kps(frame_rgb)
                H_nou=calcula_H(kps_f) if kps_f else None
                if H_nou is not None: H_actual=H_nou

            # Tracking
            results=model_tracking.track(frame,persist=True,
                tracker='bytetrack.yaml',classes=[0],conf=0.4,verbose=False)

            if results[0].boxes is not None and results[0].boxes.id is not None:
                boxes=results[0].boxes.xyxy.cpu().numpy()
                ids=results[0].boxes.id.cpu().numpy().astype(int)
                for box,tid in zip(boxes,ids):
                    cx_px=(box[0]+box[2])/2; cy_px=box[3]
                    cx_m=cy_m=None
                    if CAMERA_FIXA and H_actual is not None:
                        cx_m,cy_m=pixels_a_metres(cx_px,cy_px,H_actual)
                    roi=frame[int(box[1]):int(box[3]),int(box[0]):int(box[2])]
                    llum=float(roi.mean()) if roi.size>0 else 128
                    eq=EQUIP_LOCAL if ((COLOR_LOCAL=='clar' and llum>128) or
                                       (COLOR_LOCAL=='fosc' and llum<=128)) else EQUIP_VISITANT
                    resultats.append({'temps_seg':round(temps_seg,2),'frame':frame_idx,
                        'track_id':int(tid),'equip':eq,
                        'x_pixels':round(cx_px,1),'y_pixels':round(cy_px,1),
                        'x_metres':cx_m,'y_metres':cy_m})
            proc_idx+=1; pbar.update(1)
        frame_idx+=1
cap.release()

df_tracking=pd.DataFrame(resultats)
if CAMERA_FIXA and not df_tracking.empty:
    df_tracking=df_tracking.sort_values(['track_id','temps_seg'])
    df_tracking['dx']=df_tracking.groupby('track_id')['x_metres'].diff().fillna(0)
    df_tracking['dy']=df_tracking.groupby('track_id')['y_metres'].diff().fillna(0)
    df_tracking['dist_frame']=np.sqrt(df_tracking['dx']**2+df_tracking['dy']**2)
    df_tracking.loc[df_tracking['dist_frame']>5,'dist_frame']=0
    print('✅ Distàncies calculades en metres reals')
else:
    df_tracking['dist_frame']=0
df_tracking['jugadora']=df_tracking['track_id'].astype(str).apply(lambda x: f'Jugadora_{x}')
elapsed=time.time()-t0
print(f'\n✅ {elapsed/60:.1f} min · {len(df_tracking)} registres · {df_tracking["track_id"].nunique()} persones')


## CEL·LA 11 — Creuament accions API + posicions del vídeo

In [ ]:
df_accions=pd.DataFrame()
if not df_api.empty:
    print('Creuant API amb posicions del vídeo...')
    accions_geo=[]
    for _,row in df_api.iterrows():
        accio=row['accio']; jugadora=row['jugadora']
        if not accio: continue
        t_vid=joc_a_video(row['temps_seg'],row['quart'])
        eq_nom=equip_noms.get(row['idEquip'],'?')
        x_m=y_m=None
        if CAMERA_FIXA and not df_tracking.empty:
            df_jug=df_tracking[df_tracking['jugadora']==jugadora]
            if not df_jug.empty:
                idx_p=(df_jug['temps_seg']-t_vid).abs().idxmin()
                fila=df_jug.loc[idx_p]
                if abs(fila['temps_seg']-t_vid)<=3 and fila['x_metres'] is not None:
                    x_m,y_m=fila['x_metres'],fila['y_metres']
        tipus='altre'
        if any(c in accio for c in ['Cistella de 2','Cistella de 3','Cistella de 1']): tipus='cistella'
        elif 'Intent fallat' in accio: tipus='tir_fallat'
        elif 'falta' in accio.lower(): tipus='falta'
        elif 'Rebot' in accio: tipus='rebot'
        accions_geo.append({'temps_joc':row['temps_seg'],'temps_video':t_vid,
            'quart':row['quart'],'jugadora':jugadora,'equip':eq_nom,
            'accio':accio,'tipus':tipus,'marcador':row['marcador'],
            'x_metres':round(x_m,2) if x_m else None,
            'y_metres':round(y_m,2) if y_m else None,
            'geolocalitzat':x_m is not None})
    df_accions=pd.DataFrame(accions_geo)
    geo=df_accions['geolocalitzat'].sum(); total_a=len(df_accions)
    print(f'✅ {total_a} accions analitzades')
    if CAMERA_FIXA: print(f'   {geo}/{total_a} geolocalitzades ({geo/total_a*100:.0f}%)')
    print(); print(df_accions.groupby(['equip','tipus']).size().unstack(fill_value=0))
else:
    # Sense API: estadístiques bàsiques del tracking
    df_accions=df_tracking[['temps_seg','track_id','jugadora','equip']].copy()
    df_accions['tipus']='tracking'; df_accions['accio']='Posició'
    df_accions['marcador']=''; df_accions['quart']=0
    print(f'✅ {len(df_accions)} posicions de tracking (sense API)')


## CEL·LA 12 — Mapes i gràfics visuals

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

def pista(ax):
    ax.set_facecolor('#f0f4f8'); ax.set_aspect('equal')
    ax.set_xlim(-0.5,PISTA_LLARG+0.5); ax.set_ylim(-0.5,PISTA_AMPLE+0.5)
    ax.add_patch(patches.Rectangle((0,0),PISTA_LLARG,PISTA_AMPLE,fill=False,edgecolor='#374151',lw=2))
    ax.add_patch(patches.Rectangle((0,PISTA_AMPLE/2-3),5.8,6,facecolor='#dbeafe',edgecolor='#374151',lw=1))
    ax.add_patch(patches.Rectangle((PISTA_LLARG-5.8,PISTA_AMPLE/2-3),5.8,6,facecolor='#dbeafe',edgecolor='#374151',lw=1))
    ax.add_patch(patches.Arc((0,PISTA_AMPLE/2),27.5,27.5,angle=0,theta1=-90,theta2=90,color='#374151',lw=1.5))
    ax.add_patch(patches.Arc((PISTA_LLARG,PISTA_AMPLE/2),27.5,27.5,angle=0,theta1=90,theta2=270,color='#374151',lw=1.5))
    ax.plot(1.575,PISTA_AMPLE/2,'o',color='#374151',ms=7)
    ax.plot(PISTA_LLARG-1.575,PISTA_AMPLE/2,'o',color='#374151',ms=7)
    ax.axvline(PISTA_LLARG/2,color='#374151',lw=1.5)

equips_vid=[e for e in df_accions['equip'].unique() if e and e!='?'] if not df_accions.empty else []

# Gràfic estadístiques per equip
if not df_accions.empty and 'tipus' in df_accions.columns:
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    fig.suptitle(f'{EQUIP_LOCAL} vs {EQUIP_VISITANT} — Mode {MODE.upper()}',fontsize=13,fontweight='bold')
    for ax,eq,color in [(axes[0],equips_vid[0] if equips_vid else EQUIP_LOCAL,'#185FA5'),
                        (axes[1],equips_vid[1] if len(equips_vid)>1 else EQUIP_VISITANT,'#993C1D')]:
        df_e=df_accions[df_accions['equip']==eq]
        cats=['cistella','tir_fallat','falta','rebot']
        vals=[int((df_e['tipus']==c).sum()) for c in cats]
        bars=ax.bar(['Cistelles','Tirs fallats','Faltes','Rebots'],vals,color=color,alpha=0.8,edgecolor='white',lw=1.5)
        ax.bar_label(bars,padding=3,fontsize=11,fontweight='bold')
        ax.set_title(eq,fontsize=12,color=color,fontweight='bold')
        ax.set_facecolor('#f9fafb'); ax.spines[['top','right']].set_visible(False)
        ax.set_ylim(0,max(vals)*1.25 if max(vals)>0 else 10)
    plt.tight_layout(); plt.savefig('estadistiques_partit.png',dpi=150,bbox_inches='tight'); plt.show()
    print('✅ Gràfic estadístiques generat')

# Mapa de tir real (NOMÉS si tenim coordenades)
if CAMERA_FIXA and not df_accions.empty and df_accions.get('geolocalitzat',pd.Series()).any():
    df_tirs=df_accions[df_accions['tipus'].isin(['cistella','tir_fallat'])&df_accions['geolocalitzat']]
    if not df_tirs.empty:
        fig,axes=plt.subplots(1,2,figsize=(16,7))
        fig.suptitle(f'Mapa de tir REAL ({MODE.upper()})',fontsize=14,fontweight='bold')
        for ax,eq,ca,cf in [(axes[0],equips_vid[0] if equips_vid else EQUIP_LOCAL,'#185FA5','#93c5fd'),
                            (axes[1],equips_vid[1] if len(equips_vid)>1 else EQUIP_VISITANT,'#993C1D','#fca5a5')]:
            pista(ax)
            df_e=df_tirs[df_tirs['equip']==eq]
            cist=df_e[df_e['tipus']=='cistella']; fall=df_e[df_e['tipus']=='tir_fallat']
            ax.scatter(fall['x_metres'],fall['y_metres'],c=cf,s=80,marker='x',lw=2,alpha=0.7,label=f'Fallat ({len(fall)})',zorder=3)
            ax.scatter(cist['x_metres'],cist['y_metres'],c=ca,s=100,marker='o',edgecolors='white',lw=1.5,alpha=0.85,label=f'Cistella ({len(cist)})',zorder=4)
            ef=round(len(cist)/(len(cist)+len(fall))*100) if (len(cist)+len(fall))>0 else 0
            ax.set_title(f'{eq} — {ef}% eficiència',fontsize=12,color=ca,fontweight='bold')
            ax.legend(fontsize=10); ax.set_xlabel('Metres'); ax.set_ylabel('Metres')
        plt.tight_layout(); plt.savefig('mapa_tir_real.png',dpi=150,bbox_inches='tight'); plt.show()
        print('✅ Mapa de tir real generat!')

# Mapa de calor (NOMÉS si tenim coordenades)
if CAMERA_FIXA and not df_tracking.empty:
    df_pos=df_tracking.dropna(subset=['x_metres','y_metres'])
    if not df_pos.empty:
        fig,axes=plt.subplots(1,2,figsize=(16,7))
        fig.suptitle('Mapa de calor de posicions',fontsize=14,fontweight='bold')
        for ax,eq,cmap,color in [(axes[0],EQUIP_LOCAL,'Blues','#185FA5'),(axes[1],EQUIP_VISITANT,'Reds','#993C1D')]:
            pista(ax)
            df_e=df_pos[df_pos['equip']==eq]
            if not df_e.empty:
                h=ax.hist2d(df_e['x_metres'],df_e['y_metres'],bins=[28,15],range=[[0,28],[0,15]],cmap=cmap,alpha=0.65)
                plt.colorbar(h[3],ax=ax,label='Presència')
            ax.set_title(eq,fontsize=12,color=color,fontweight='bold')
        plt.tight_layout(); plt.savefig('mapa_calor.png',dpi=150,bbox_inches='tight'); plt.show()
        print('✅ Mapa de calor generat!')

if not CAMERA_FIXA:
    print()
    print('ℹ️  Mapa de tir i mapa de calor no disponibles (sense coordenades reals).')
    print('   Per activar-los:')
    print('   - Càmera fixa: el notebook els detectarà automàticament')
    print('   - Càmera mòbil: canvia MODE="advanced" i afegeix una API key de Roboflow')


## CEL·LA 13 — Estadístiques per jugadora

In [ ]:
resum=df_tracking.groupby(['track_id','equip']).agg(
    aparicions=('temps_seg','count'),
    temps_min=('temps_seg','min'),
    temps_max=('temps_seg','max'),
).reset_index()
resum['minuts_visibles']=((resum['temps_max']-resum['temps_min'])/60).round(1)

if CAMERA_FIXA:
    dist=df_tracking.groupby('track_id')['dist_frame'].sum().reset_index()
    dist.columns=['track_id','metres_total']
    dist['metres_total']=dist['metres_total'].round(0)
    resum=resum.merge(dist,on='track_id',how='left')
    resum['m_per_min']=(resum['metres_total']/resum['minuts_visibles'].replace(0,1)).round(1)

if not df_api.empty:
    pts_jug=df_api.groupby('jugadora')['accio'].apply(
        lambda x: int(x.str.contains('Cistella de 3',na=False).sum()*3+
                     x.str.contains('Cistella de 2',na=False).sum()*2+
                     x.str.contains('Cistella de 1',na=False).sum())).reset_index(name='punts')
    resum=resum.merge(pts_jug,left_on='track_id',right_on='jugadora',how='left').fillna(0)

print('📊 Estadístiques per ID de tracking:')
cols=['track_id','equip','minuts_visibles']
if CAMERA_FIXA and 'metres_total' in resum.columns: cols+=['metres_total','m_per_min']
if 'punts' in resum.columns: cols.append('punts')
print(resum[cols].sort_values('minuts_visibles',ascending=False).head(15).to_string(index=False))


## CEL·LA 14 — Exporta els CSV per a Miki Analítica

In [ ]:
from google.colab import files
import os
exports=[]

# CSV tracking
f1=f'{NOM_SORTIDA}_tracking.csv'
cols_exp=['temps_seg','frame','track_id','jugadora','equip','x_pixels','y_pixels']
if CAMERA_FIXA: cols_exp+=['x_metres','y_metres','dist_frame']
df_tracking[cols_exp].to_csv(f1,index=False); exports.append(f1)

# CSV resum jugadores
f2=f'{NOM_SORTIDA}_resum.csv'
resum.to_csv(f2,index=False); exports.append(f2)

# CSV accions
if not df_accions.empty:
    f3=f'{NOM_SORTIDA}_accions.csv'
    df_accions.to_csv(f3,index=False); exports.append(f3)

# Imatges generades
for img in ['estadistiques_partit.png','mapa_tir_real.png','mapa_calor.png']:
    if os.path.exists(img): exports.append(img)

print(f'Mode: {MODE.upper()} · Càmera: {"FIXA" if CAMERA_FIXA else "MÒBIL"}')
print('Descarregant fitxers...')
for f in exports:
    try: files.download(f); print(f'  ✅ {f}')
    except: print(f'  ⚠️  {f} no trobat')
print('\n🏀 Importa els CSV a Miki Analítica!')
